### Reconstruction of an accidentally discarded notebook used to join PTB-XL and PTB-XL+ databases

The original notebook saved the resulting file and it still exists at: `/standard/ds7200-apt4c/team2/ptbxl_nw`

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

base_team2 = "/standard/ds7200-apt4c/team2"
base_xlplus = f"{base_team2}/ptb-xl-a-comprehensive-electrocardiographic-feature-dataset-1.0.1/features"
base_xl = f"{base_team2}/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

ptbxl = spark.read.csv(f"{base_xl}/ptbxl_database.csv", header=True, inferSchema=True)
sl12 = spark.read.csv(f"{base_xlplus}/12sl_features.csv", header=True, inferSchema=True)
ecgdeli = spark.read.csv(f"{base_xlplus}/ecgdeli_features.csv", header=True, inferSchema=True)
unig = spark.read.csv(f"{base_xlplus}/unig_features.csv", header=True, inferSchema=True)

ptbxl_trim = ptbxl.drop("nurse", "site", "device", "validated_by", "recording_date")

def add_prefix(df, prefix, key="ecg_id"):
    return df.select(key, *[df[c].alias(f"{prefix}_{c}") for c in df.columns if c != key])

sl12_p = add_prefix(sl12, "sl12")
ecgdeli_p = add_prefix(ecgdeli, "ecgdeli")
unig_p = add_prefix(unig, "unig")

ptbxl_nw = (ptbxl_trim
            .join(sl12_p, on="ecg_id", how="inner")
            .join(ecgdeli_p, on="ecg_id", how="inner")
            .join(unig_p, on="ecg_id", how="inner"))